In [1]:
import random
import numpy as np
import matplotlib.pyplot as plt

In [2]:

mutation_rates = [0.2, 0.5, 1.0]
crossover_rates = [0.2, 0.5, 1.0]
n_values = [10, 12, 20]
strategies = ['Elitism', 'Generational']

In [3]:

class GeneticAlgorithm:
    def __init__(self, n, population_size, mutation_rate, recombination_rate, generations, elitism=False):
        self.n = n
        self.population_size = population_size
        self.mutation_rate = mutation_rate
        self.recombination_rate = recombination_rate
        self.generations = generations
        self.elitism = elitism
        self.population = []
        self.max_fitness = []
        self.avg_fitness = []

    def initialize_population(self):
        self.population = [self.random_individual() for _ in range(self.population_size)]
        self.evaluate_population()

    def random_individual(self):
        return list(np.random.choice(range(1, self.n + 1), self.n, replace=False)) + [None]

    def fitness(self, individual):
        encounter = 0
        for i in range(self.n):
            for j in range(i + 1, self.n):
                if individual[i] == individual[j] or abs(i - j) == abs(individual[i] - individual[j]):
                    encounter += 1
        return encounter

    def evaluate_population(self):
        for individual in self.population:
            individual[-1] = self.fitness(individual[:-1])

    def evaluate_individuals(self, individuals):
        for individual in individuals:
            individual[-1] = self.fitness(individual[:-1])

    def select_parents(self):
        parents = random.sample(self.population, 5)
        best_two = sorted(parents, key=lambda x: x[-1])
        return [best_two[0], best_two[1]]

    def crossover(self, parents):
        parent1_genes = parents[0][:-1]
        parent2_genes = parents[1][:-1]
        if random.random() < self.recombination_rate:
            cut = random.randint(0, self.n - 1)
            child1 = parent1_genes[:cut] + [i for i in parent2_genes if i not in parent1_genes[:cut]]
            child2 = parent2_genes[:cut] + [i for i in parent1_genes if i not in parent2_genes[:cut]]
        else:
            child1, child2 = parent1_genes[:], parent2_genes[:]
        return [child1 + [None], child2 + [None]]

    def swap_mutation(self, offspring):
        for child in offspring:
            if random.random() < self.mutation_rate:
                idx1, idx2 = random.sample(range(self.n), 2)
                child[idx1], child[idx2] = child[idx2], child[idx1]
        return offspring

    def generational_replacement(self, offspring):
        self.population = offspring
        self.evaluate_population()

    def elitism_replacement(self, offspring):
        self.evaluate_individuals(offspring)
        offspring = sorted(offspring, key=lambda x: x[-1])
        self.population = sorted(self.population, key=lambda x: x[-1])
        self.population[-2:] = offspring[:2]
        self.evaluate_population()

    def run(self):
        self.initialize_population()
        for gen in range(self.generations):
            offspring = []
            for _ in range(self.population_size // 2):
                parents = self.select_parents()
                children = self.crossover(parents)
                children = self.swap_mutation(children)
                self.evaluate_individuals(children)
                offspring.extend(children)

            if self.elitism:
                self.elitism_replacement(offspring)
            else:
                self.generational_replacement(offspring)

            best_individual = min(self.population, key=lambda ind: ind[-1])
            best_fitness = best_individual[-1]
            average_fitness = sum(ind[-1] for ind in self.population) / self.population_size

            self.max_fitness.append(best_fitness)
            self.avg_fitness.append(average_fitness)
            
            print(f'{gen} : Best F : {best_fitness} | AVG : {average_fitness} | {best_individual} | len : {len(best_individual)}')
            

            if best_fitness == 0:
                print(f"Solution found at generation {gen}, Best Gene: {best_individual[:-1]}")
                self.display_board(best_individual[:-1])
                break
            

    
    def display_board(self, individual):
        from matplotlib.patches import Rectangle
        fig, ax = plt.subplots(figsize=(6, 6))
        
  
        light_color = '#F0D9B5' 
        dark_color = '#B58863'   
        
        for i in range(self.n):
            for j in range(self.n):
                rect = Rectangle((j, self.n - i - 1), 1, 1, facecolor=(light_color if (i + j) % 2 == 0 else dark_color))
                ax.add_patch(rect)
        
  
        for row, col in enumerate(individual):
            ax.text(col - 1 + 0.5, self.n - row - 1 + 0.5, '♛', ha='center', va='center', fontsize=24, color='black')
        
        ax.set_xlim(0, self.n)
        ax.set_ylim(0, self.n)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_aspect('equal')
        plt.title('Queen Positions on Chessboard', fontsize=16)
        plt.show()


    def plot_fitness(self, mutation_rate, crossover_rate, strategy):
        plt.figure(figsize=(10, 6))
        plt.style.use('dark_background')
        generations = list(range(len(self.max_fitness)))
        plt.plot(generations, self.max_fitness, label='Max Fitness', color='#00FFEA', linewidth=2, marker='o', markersize=6, alpha=0.8)
        plt.plot(generations, self.avg_fitness, label='Average Fitness', color='#FFD700', linestyle='--', linewidth=2, marker='s', markersize=6, alpha=0.8)
        plt.title(f'Fitness Progression (N={self.n}, Mutation={mutation_rate}, Crossover={crossover_rate}, Strategy={strategy})', fontsize=12)
        plt.xlabel('Generation', fontsize=12)
        plt.ylabel('Fitness', fontsize=12)
        plt.xticks(fontsize=10)
        plt.yticks(fontsize=10)
        plt.grid(True, color='#555555', linestyle='-', alpha=0.3)
        plt.legend(loc='best', fontsize=10)
        plt.tight_layout()
        plt.show()

In [ ]:
for n in n_values:
    for mutation_rate in mutation_rates:
        for crossover_rate in crossover_rates:
            for strategy in strategies:
                elitism = (strategy == 'Elitism')
                print(f"Running GA for N={n}, Mutation Rate={mutation_rate}, Crossover Rate={crossover_rate}, Strategy={strategy}")
                ga = GeneticAlgorithm(n=n, population_size=100, mutation_rate=mutation_rate, recombination_rate=crossover_rate, generations=10000, elitism=elitism)
                ga.run()
                ga.plot_fitness(mutation_rate, crossover_rate, strategy)